In [ ]:
import os
os.environ.pop("PYTHONHASHSEED", None)

!pip install -q fastapi uvicorn soundfile f5-tts safetensors torch numpy ruaccent pyngrok nest-asyncio openai-whisper

import re
import json
import tempfile
import urllib.request
import asyncio
import soundfile as sf
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Response
import uvicorn
from f5_tts.api import F5TTS
import safetensors.torch as sf_torch
import numpy as np
from ruaccent import RUAccent
import whisper
from pyngrok import ngrok
import nest_asyncio

nest_asyncio.apply()

app = FastAPI()

print("🚀 Инициализация RUAccent, Whisper и F5-TTS (v2) на GPU...")

accentizer = RUAccent()
accentizer.load(
    omograph_model_size='turbo3.1',
    use_dictionary=True,
    tiny_mode=False,
    device="cuda"
)
print("✅ RUAccent успешно загружен!")

print("📥 Загрузка модели Whisper (small)...")
whisper_model = whisper.load_model("small", device="cuda")
print("✅ Whisper успешно загружен!")

if not os.path.exists("vocab.txt"):
    os.system("wget -q https://huggingface.co/Misha24-10/F5-TTS_RUSSIAN/resolve/main/F5TTS_v1_Base/vocab.txt -O vocab.txt")

v2_safetensors_url = "https://huggingface.co/Misha24-10/F5-TTS_RUSSIAN/resolve/main/F5TTS_v1_Base_v2/model_last_inference.safetensors"
if not os.path.exists("model_last_inference_v2.safetensors"):
    os.system(f"wget -q {v2_safetensors_url} -O model_last_inference_v2.safetensors")

clean_ckpt_path = "model_cleaned_inference_v2.safetensors"
if not os.path.exists(clean_ckpt_path):
    state_dict = sf_torch.load_file("model_last_inference_v2.safetensors")
    new_state_dict = {k.replace("ema_model.", ""): v for k, v in state_dict.items() if k.startswith("ema_model.")}
    if not new_state_dict:
        new_state_dict = {k: v for k, v in state_dict.items() if k != "step"}
    sf_torch.save_file(new_state_dict, clean_ckpt_path)

TAG_PATTERN = re.compile(r"\[.*?\]\s*")

def clean_tags(text: str) -> str:
    cleaned = TAG_PATTERN.sub("", text)
    return re.sub(r'\s+', ' ', cleaned).strip()

CHARACTERS_CONFIG = {
    "sofia": {
        "excited": {"filename": "excited.mp3", "text": "[excited, cheerful] Слушай, представляешь! [chuckles] Я раньше так сильно увлекалась мемами, что мы с подругой даже свой паблик вели!"},
        "sad": {"filename": "sad.mp3", "text": "[sad, slow] Ты знаешь... [sighs] И на этом все наши хиханьки-хаханьки закончились на длительное время"},
        "ironic": {"filename": "ironic.mp3", "text": "[sarcastic, calm] На самом деле я сейчас живу в пристройке у родителей, потому что в своей квартире я делаю ремонт. [sighs] Ремонта там, конечно, очень много... [pause] Ну просто прелесть, а не жизнь."},
        "flirty": {"filename": "flirty.mp3", "text": "[playful, cheerful] Знаешь, я тут подумала..."},
        "surprised": {"filename": "surprised.mp3", "text": "[surprised, excited] Подожди-ка... [gasp] Ты хочешь сказать, что мы действительно собрали всю эту коллекцию с нуля и никто даже не заметил?! [chuckles] Это просто невероятно!"},
        "angry": {"filename": "angry.mp3", "text": "[annoyed, angry] Сколько можно повторять одно и то же?! [sighs] Я уже битый час пытаюсь объяснить элементарные вещи, а вы снова за своё!"},
        "tired": {"filename": "tired.mp3", "text": "[tired, calm] Ох... как же я устала сегодня... [sighs] Весь день этот ремонт, дела, куча забот..."},
        "passionate": {"filename": "passionate.mp3", "text": "[sensual, whispers] Ах... как же жарко сегодня... [sighs] Подойди ближе ко мне, я хочу сказать тебе кое-что очень тихо... [breathing] Ты даже не представляешь, как сильно я этого ждала... [gasp]"},
        "moan_cheerful": {"filename": "moan_cheerful.mp3", "text": "[sensual, whispers] Ох... [moan] как же это приятно... [sighs] Продолжай, не останавливайся... [gasp] мне так хорошо с тобой..."},
    }
}

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/ChronoMonochrome/AIGurl/refs/heads/master/voices"
LOADED_CHARACTERS = {}

for char_name, refs in CHARACTERS_CONFIG.items():
    char_dir = os.path.join("voices", char_name)
    os.makedirs(char_dir, exist_ok=True)
    LOADED_CHARACTERS[char_name] = {}
    for ref_key, data in refs.items():
        local_path = os.path.join(char_dir, data["filename"])
        if not os.path.exists(local_path):
            file_url = f"{GITHUB_RAW_BASE}/{char_name}/{data['filename']}"
            try:
                urllib.request.urlretrieve(file_url, local_path)
            except Exception as e:
                print(f"⚠️ Ошибка скачивания {file_url}: {e}")
        if os.path.exists(local_path):
            LOADED_CHARACTERS[char_name][ref_key] = {
                "file": local_path,
                "text": clean_tags(data["text"])
            }

f5tts = F5TTS(
    model="F5TTS_v1_Base",
    ckpt_file=clean_ckpt_path,
    vocab_file="vocab.txt",
    use_ema=False,
    device="cuda"
)
print("✅ F5-TTS v2 и Whisper полностью готовы к работе в дуплексном режиме!")

@app.post("/tts")
async def tts_endpoint(data: dict):
    raw_text = data.get("text", "")
    requested_char = data.get("character", "sofia")
    requested_ref = data.get("ref", "excited")
    if not raw_text:
        return {"error": "No text provided"}

    char_refs = LOADED_CHARACTERS.get(requested_char, LOADED_CHARACTERS["sofia"])
    ref_data = char_refs.get(requested_ref, char_refs["excited"])

    # Безопасная обработка акцентов (при сбое ONNX просто используем чистый текст)
    try:
        processed_text = accentizer.process_all(raw_text)
    except Exception as e:
        print(f"⚠️ Ошибка RUAccent (пропускаем акценты): {e}")
        processed_text = raw_text

    word_count = len(raw_text.split())
    nfe_steps = 16 if word_count <= 3 else 32

    try:
        wav, sr, _ = f5tts.infer(
            ref_file=ref_data["file"],
            ref_text=ref_data["text"],
            gen_text=processed_text,
            nfe_step=nfe_steps,
            cfg_strength=2.0,
        )
        padding = np.zeros(int(sr * 0.2), dtype=wav.dtype)
        wav = np.concatenate([wav, padding])

        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        sf.write(temp_file.name, wav, sr)
        temp_file.close()

        with open(temp_file.name, "rb") as f:
            audio_bytes = f.read()
        os.remove(temp_file.name)
        return Response(content=audio_bytes, media_type="audio/wav")
    except Exception as e:
        print(f"❌ Ошибка синтеза F5-TTS: {e}")
        return {"error": str(e)}, 500)

@app.websocket("/ws/voice")
async def voice_websocket(websocket: WebSocket):
    await websocket.accept()
    print("🔌 Клиент подключен по WebSocket (дуплекс)")

    audio_buffer = bytearray()
    # Динамический контекст диалога для Whisper
    transcription_history = []

    try:
        while True:
            try:
                message = await asyncio.wait_for(websocket.receive(), timeout=30.0)
            except asyncio.TimeoutError:
                await websocket.send_json({"type": "ping"})
                continue

            if "bytes" in message:
                audio_buffer.extend(message["bytes"])

            elif "text" in message:
                data = json.loads(message["text"])
                action = data.get("action")

                if action == "speech_end":
                    if len(audio_buffer) > 0:
                        temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
                        temp_name = temp_audio.name
                        temp_audio.close()

                        audio_np = np.frombuffer(audio_buffer, dtype=np.int16).astype(np.float32) / 32768.0
                        sf.write(temp_name, audio_np, 16000)
                        audio_buffer.clear()

                        # Собираем контекст из последних реплик для динамического промпта Whisper
                        dynamic_prompt = "Разговор Дениса и девушки. " + " ".join(transcription_history[-6:])

                        print("🎙️ Распознавание речи через Whisper (с контекстом)...")
                        result = whisper_model.transcribe(
                            temp_name,
                            language="ru",
                            initial_prompt=dynamic_prompt,
                            condition_on_previous_text=False,
                            beam_size=5
                        )
                        user_text = result.get("text", "").strip()

                        if os.path.exists(temp_name):
                            os.remove(temp_name)

                        print(f"📝 Распознанный текст: '{user_text}'")

                        # Добавляем реплику пользователя в историю контекста
                        if user_text:
                            transcription_history.append(f"Денис: {user_text}")

                        await websocket.send_json({
                            "type": "transcription",
                            "text": user_text
                        })
                elif action == "assistant_said":
                    # Клиент сообщает серверу, что ответила девушка, чтобы Whisper знал об этом
                    bot_text = data.get("text", "")
                    if bot_text:
                        transcription_history.append(f"София: {bot_text}")
                elif action == "clear_buffer" or action == "interrupt":
                    audio_buffer.clear()
                    print("⚡ Буфер аудио на сервере очищен по сигналу прерывания.")

    except WebSocketDisconnect:
        print("🔌 Клиент отключился от WebSocket")
    except Exception as e:
        print(f"⚠️ Ошибка в WebSocket: {e}")

NGROK_TOKEN = ""
STATIC_URL = ""

try:
    ngrok.set_auth_token(NGROK_TOKEN)
except:
    pass

public_url = ngrok.connect(8000, url=STATIC_URL).public_url
print(f"\n🌐 Статический туннель ngrok успешно запущен: {public_url}\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()